In [18]:
from pathlib import Path

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

from sklearn.preprocessing import PowerTransformer, RobustScaler

In [11]:
root = Path("/data/xujiayi/xjy/")

dates = np.load(root / "axis" / "dates.npy", allow_pickle=True)
ticks = np.load(root / "axis" / "ticks.npy", allow_pickle=True)

T, N = len(dates), len(ticks)

pct = np.memmap(
    root / "d_field" / "pct.bin",
    mode="r",
    shape=(T, N),
    dtype=float,
)

window = 10
topk = min(50, N - 1)
eps = 1e-12

In [12]:
future_view = sliding_window_view(
    1.0 + pct,
    window_shape=window,
    axis=0,
)  # [T-window+1, N, window]

y10 = np.full((T, N), np.nan, dtype=np.float32)

y10[: T - window + 1] = (
    np.prod(future_view, axis=-1) - 1.0
)

y10[:-2] = y10[2:]
y10[-2:]=np.nan

In [23]:
top50_idx = np.memmap(
    root / "label" / "pct_corr_top50_idx.bin",
    mode="w+",
    shape=(T, N, topk),
    dtype=np.int32,
)

y10_zscore = np.memmap(
    root / "label" / "y10_peer_zscore.bin",
    mode="w+",
    shape=(T, N),
    dtype=np.float32,
)

peer_mean_arr = np.memmap(
    root / "label" / "peer_mean_arr.bin",
    mode="w+",
    shape=(T, N),
    dtype=np.float32,
)

top50_idx[:] = -1
y10_zscore[:] = np.nan
peer_mean_arr[:] = np.nan


In [22]:
past_view = sliding_window_view(
    pct,
    window_shape=window,
    axis=0,
)  # [T-window+1, N, window]

In [6]:
ret10 = np.full((T, N), np.nan, dtype=np.float32)

ret10[window - 1:] = (
    np.prod(
        sliding_window_view(
            1.0 + pct,
            window_shape=window,
            axis=0,
        ),
        axis=-1,
    )
    - 1.0
)

past_view = sliding_window_view(
    ret10,
    window_shape=window,
    axis=0,
)

In [ ]:
for k, x in enumerate(past_view):

    t = k + window - 1

    x = np.asarray(x, dtype=np.float32)

    # 过去10日数据完整的股票
    valid = np.isfinite(x).all(axis=1)

    # 有效股票不足51只，无法为每只股票选择50只其他股票
    if valid.sum() <= topk:
        continue

    # 无效行填0，避免NaN参与矩阵运算
    x = np.where(valid[:, None], x, 0.0)

    # 去均值并单位化
    x -= x.mean(axis=1, keepdims=True)

    norm = np.linalg.norm(x, axis=1)
    valid &= norm > eps

    x = np.divide(
        x,
        norm[:, None],
        out=np.zeros_like(x),
        where=norm[:, None] > eps,
    )

    # Pearson相关系数矩阵
    corr = x @ x.T

    # 无效股票不能作为候选股票
    corr[:, ~valid] = -np.inf

    # 排除自身
    np.fill_diagonal(corr, -np.inf)

    # 每行选择相关性最大的50只股票，不进行内部排序
    idx = np.argpartition(
        corr,
        kth=N - topk,
        axis=1,
    )[:, -topk:]

    # 只有目标股票自身有效时，才保存Top50
    top50_idx[t, valid] = idx[valid]

    # 取Top50股票当日的未来10日收益
    peer_y10 = y10[t, idx]  # [N, topk]

    # 自动忽略Top50中y10为nan的股票
    peer_mean = np.nanmean(peer_y10, axis=1)
    peer_mean_arr[t] = peer_mean
    peer_std = np.nanstd(peer_y10, axis=1)

    # 用Top50股票的均值、标准差标准化自己的y10
    can_calc = (
        valid
        & np.isfinite(y10[t])
        & np.isfinite(peer_mean)
        & (peer_std > eps)
    )

    y10_zscore[t] = np.divide(
        y10[t] - peer_mean,
        peer_std,
        out=np.full(N, np.nan, dtype=np.float32),
        where=can_calc,
    )

/tmp/ipykernel_1208922/585521904.py:53: RuntimeWarning: Mean of empty slice
  peer_mean = np.nanmean(peer_y10, axis=1)
/home/xujiayi/PycharmProjects/Models/PINN-MTICG/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [35]:
peer_mean_arr.flush()
top50_idx.flush()
y10_zscore.flush()

print("计算完成")
print("y10:", y10.shape)
print("top50_idx:", top50_idx.shape)
print("y10_zscore:", y10_zscore.shape)

计算完成
y10: (4376, 5436)
top50_idx: (4376, 5436, 50)
y10_zscore: (4376, 5436)


In [31]:
y10_zscore[8:-8]

memmap([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.06591698,  0.16253407, -0.48439237, ...,         nan,
                 nan,         nan],
        [ 0.4522234 ,  0.524777  , -0.7315957 , ...,         nan,
                 nan,         nan],
        ...,
        [-0.6776986 , -1.3175482 ,  0.73919624, ...,         nan,
         -0.45172757,  0.9273144 ],
        [-0.83280015, -0.9764051 ,  0.9200718 , ...,         nan,
         -0.33316013,  0.611081  ],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan]], shape=(4360, 5436), dtype=float32)

In [22]:
y10_zscore[y10_zscore!=0]

array([ 0.06591698,  0.16253407, -0.48439237, ...,         nan,
               nan,         nan], shape=(23739012,), dtype=float32)

In [54]:
y10_zscore.astype(float).tofile( root / "label" / "y10_peer_zscore.bin")

In [3]:
import bottleneck as bn

In [19]:
class Processor:

    @staticmethod
    def calc_indmv_neutral_longshort(ind_signal, temp_mv):
        ix = ~(np.isnan(ind_signal) | np.isinf(ind_signal) | np.isnan(temp_mv) | np.isinf(temp_mv))
        ind_signal[~ix] = np.nan
        temp_mv[~ix] = np.nan

        mv_mean = bn.nanmean(temp_mv, axis=1)
        signal_mean = bn.nanmean(ind_signal, axis=1)
        m = (mv_mean * signal_mean - bn.nanmean(temp_mv * ind_signal, axis=1)) / (mv_mean**2 - bn.nanmean(temp_mv**2, axis=1) + 1e-6)
        b = signal_mean - m * mv_mean
        residual = (ind_signal.T - (temp_mv.T * m) - b).T
        ind_signal = (residual.T - bn.nanmean(residual, axis=1)) / (bn.nanstd(residual, axis=1) + 1e-6)
        return ind_signal.T

    @staticmethod
    def indmv_neutral_longshort(alpha_vec, ind_arr, mv_arr):
        new_signal = np.full_like(alpha_vec, np.nan)   # [T,N]
        ln_mv = np.log(mv_arr)
        for i in range(31):
            ind_ix = ind_arr == i
            ind_select = ind_ix.any(axis=0)
            ind_ix_select = ind_ix[:, ind_select]
            ind_signal = alpha_vec[:, ind_select].copy()
            ind_signal[~ind_ix_select] = np.nan
            temp_mv = ln_mv[:, ind_select].copy()
            new_signal[ind_ix] = Processor.calc_indmv_neutral_longshort(ind_signal, temp_mv)[ind_ix_select]
        return new_signal

    @staticmethod
    def cross_standardize(x):
        mean = np.nanmean(x, axis=1, keepdims=True)
        std = np.nanstd(x, axis=1, keepdims=True)
        std = np.where((std < 1e-8) | np.isnan(std), np.nan, std)
        x = np.divide(x-mean, std, out=np.full_like(x, 1), where=std!=0)
        return x
    
    @staticmethod
    def yeojohnson(x: np.ndarray) -> np.ndarray:
            pt = PowerTransformer(method='yeo-johnson')
            x = x.copy()
            mask = np.isnan(x).all(axis=1)
            arr = x[~mask]
            res = pt.fit_transform(arr.T).T
            x[~mask] = res
            return x

In [9]:
mv = np.memmap(
    root / "d_field" / "mv.bin",
    mode="r",
    shape=(T, N),
    dtype=float,
)
ind = np.memmap(
    root / "mask" / "industry.bin",
    mode="r",
    shape=(T, N),
    dtype=float,
)
ind

memmap([[25.,  4.,  4., ..., nan, nan, nan],
        [25.,  4.,  4., ..., nan, nan, nan],
        [25.,  4.,  4., ..., nan, nan, nan],
        ...,
        [25.,  4., 14., ..., 11., 17., 30.],
        [25.,  4., 14., ..., 11., 17., 30.],
        [25.,  4., 14., ..., 11., 17., 30.]], shape=(4376, 5436))

In [20]:
y10_neutral = Processor.indmv_neutral_longshort(y10, ind, mv)
y10_neutral = Processor.yeojohnson(y10_neutral)

In [52]:
y10_neutral = y10 - peer_mean_arr

In [53]:
y10_neutral.astype(float).tofile(root / "label" / "y10_neutral.bin")